# OpenPlaning model explorer

What `PlaningBoat` (from [python-openplaning](https://github.com/elcf/python-openplaning)) actually
takes as input and gives back as output, split into **hull** and **propulsion** - see the discussion
this came out of for the full breakdown. This is a reference/exploration notebook for the *library*
itself, not this project's hull: for that, see [`explore.ipynb`](explore.ipynb).

The base case below is the library's own documented Savitsky '76 worked example (from its README), not
this project's RIB - it's a known-good reference with published expected output, so the first cell below
doubles as a check that the install is working before you plug in real numbers.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="pkg_resources")   # OpenPlaning imports it; harmless

import numpy as np
import matplotlib.pyplot as plt
from openplaning import PlaningBoat

## 2. The reference boat, inputs grouped by hull vs propulsion

Everything here is the Savitsky '76 example straight from the openplaning README - a large monohull
patrol-boat-sized vessel, not this project's hull. `speed` is listed on its own because it's the shared
operating condition both sides are evaluated at, not a property of either.

In [ ]:
speed = 13.07   # m/s - SHARED: the condition both hull and propulsion are evaluated at

# --- HULL: geometry & mass ---
weight = 827400       # N
beam = 7.315           # m
length = 24.38         # m, vessel LOA
lcg = 10.67            # m, long. center of gravity, from stern
vcg = beam / 7         # m, vert. center of gravity, from keel
r_g = 0.25 * length    # m, radius of gyration
beta = 15              # deg, deadrise

# --- HULL: trim flap (a hull appendage, not propulsion) ---
sigma = 1.0    # flap span / beam
delta = 5      # deg, flap deflection
Lf = 0.3048    # m, flap chord

# --- HULL: seaway, for the seakeeping outputs later ---
H_sig = 1.402  # m, significant wave height

# --- HULL: which empirical method to use - solver mode, not physics ---
wetted_lengths_type = 3      # Savitsky '76 approach
roughness_penalty_type = 2   # Townsin '84 regression

# --- PROPULSION: thrust line only. There is no propeller/motor model in this library at all -
# thrust_force below is just "whatever the hull side demands," not what a real drive can deliver. ---
epsilon = 0    # deg, thrust angle w.r.t. keel
vT = vcg       # m, thrust vertical distance
lT = lcg       # m, thrust horizontal distance

## 3. Build it and solve for equilibrium

`get_steady_trim()` is the one call that mutates the boat: it root-finds `tau` and `z_wl` until the
hull's forces balance, starting from the constructor's defaults (`tau=5`, `z_wl=0`) as its initial
guess. Everything else below is read-only once computed.

The warnings below are expected, not a bug - this reference boat sits right at the edge of a few of the
empirical methods' documented applicability ranges (trim, Froude number, wave-height/beam), and the
library says so rather than silently extrapolating without comment.

In [ ]:
boat = PlaningBoat(
    speed, weight, beam, lcg, vcg, r_g, beta, epsilon, vT, lT, length, H_sig,
    Lf=Lf, sigma=sigma, delta=delta,
    wetted_lengths_type=wetted_lengths_type, roughness_penalty_type=roughness_penalty_type,
)
boat.get_steady_trim()
boat.print_description()

## 4. Pulling the outputs apart: hull vs propulsion

Same object, read two ways. The hull side is almost everything - geometry, forces, attitude, dynamics,
seakeeping. The propulsion side is just one vector: the thrust `get_forces()` says is required to balance
the hull, then arithmetic'd into power/horsepower. Nothing here models whether a real motor and
propeller could actually deliver it.

In [ ]:
print("HULL outputs")
print(f"  wetted geometry: L_K={boat.L_K:.3f} m, L_C={boat.L_C:.3f} m, "
      f"lambda={boat.lambda_W:.3f}, T={boat.T:.3f} m, area={boat.wetted_bottom_area:.3f} m^2")
print(f"  equilibrium attitude: z_wl={boat.z_wl:.4f} m, tau={boat.tau:.3f} deg")
print(f"  net force balance [Fx, Fz, Mcg]: {boat.net_force}")

print()
print("PROPULSION outputs")
thrust_mag = np.hypot(boat.thrust_force[0], boat.thrust_force[1])
eff_thrust = -boat.thrust_force[0]
print(f"  required thrust vector [Fx, Fz, Mcg]: {boat.thrust_force}")
print(f"  thrust magnitude: {thrust_mag:.1f} N, effective thrust: {eff_thrust:.1f} N")
print(f"  effective power: {eff_thrust * speed / 1000:.1f} kW "
      f"({eff_thrust * speed / 1000 / 0.7457:.1f} hp)")

## 5. Does a hull input move a propulsion output? Sweep speed

Hold the hull fixed, change only `speed`, re-solve equilibrium at each point, and watch what moves on
both sides.

In [ ]:
speeds = np.linspace(5, 20, 16)   # m/s
powers_kw, trims_deg = [], []
for v in speeds:
    b = PlaningBoat(v, weight, beam, lcg, vcg, r_g, beta, epsilon, vT, lT, length, H_sig,
                     Lf=Lf, sigma=sigma, delta=delta,
                     wetted_lengths_type=wetted_lengths_type, roughness_penalty_type=roughness_penalty_type)
    b.get_steady_trim()
    eff_t = -b.thrust_force[0]
    powers_kw.append(eff_t * v / 1000)
    trims_deg.append(b.tau)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(speeds, powers_kw, marker="o")
ax1.set_xlabel("Speed (m/s)"); ax1.set_ylabel("Effective power (kW)")
ax1.set_title("PROPULSION output")
ax2.plot(speeds, trims_deg, marker="o", color="tab:orange")
ax2.set_xlabel("Speed (m/s)"); ax2.set_ylabel("Equilibrium trim (deg)")
ax2.set_title("HULL output")
fig.tight_layout()

## 6. Does a hull input move a hull output, holding speed fixed? Sweep deadrise

Same idea, but changing a pure hull-geometry input (`beta`) at a fixed speed - both a hull output
(wetted area) and the propulsion output it drives (required power) move together.

In [ ]:
betas = np.linspace(10, 25, 16)   # deg
areas, powers_kw2 = [], []
for b_deg in betas:
    b = PlaningBoat(speed, weight, beam, lcg, vcg, r_g, b_deg, epsilon, vT, lT, length, H_sig,
                     Lf=Lf, sigma=sigma, delta=delta,
                     wetted_lengths_type=wetted_lengths_type, roughness_penalty_type=roughness_penalty_type)
    b.get_steady_trim()
    eff_t = -b.thrust_force[0]
    areas.append(b.wetted_bottom_area)
    powers_kw2.append(eff_t * speed / 1000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(betas, areas, marker="o")
ax1.set_xlabel("Deadrise (deg)"); ax1.set_ylabel("Wetted bottom area (m\u00b2)")
ax1.set_title("HULL output")
ax2.plot(betas, powers_kw2, marker="o", color="tab:green")
ax2.set_xlabel("Deadrise (deg)"); ax2.set_ylabel(f"Effective power at {speed:.1f} m/s (kW)")
ax2.set_title("PROPULSION output")
fig.tight_layout()

## Next steps

To explore *this project's* hull instead of the Savitsky reference boat, use
[`explore.ipynb`](explore.ipynb) - it already builds a prismatic hull from `optimization2.md`'s
numbers and runs it through the same `PlaningBoat` calls. This notebook is deliberately kept generic so
it stays a correct reference for the library itself even if the project's own hull parameters change.

See [`README.md`](README.md) for how the finalops optimizer uses (and doesn't yet use) these outputs,
and the [openplaning GitHub repo](https://github.com/elcf/python-openplaning) for the underlying
Savitsky/Faltinsen references.